# 🎵 Messy Mashup — Music Genre Classification

Three models trained on precomputed mel spectrograms, ensembled for final submission.

| Model | Type | Notes |
|-------|------|-------|
| GenreCNN | From scratch | 5-block CNN baseline |
| EfficientNet-B0 | Pretrained | 2-phase fine-tuning |
| CRNN | From scratch | CNN + Bidirectional GRU |

> **Metric:** Macro-averaged F1 across 10 genres.


## 1. Setup

Install dependencies, authenticate W&B, and detect device.

In [1]:
# ── Install / imports ──────────────────────────────────────────────────
!pip install wandb --quiet

import os, random, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import librosa
from tqdm import tqdm
from sklearn.metrics import f1_score
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import models
import wandb
from kaggle_secrets import UserSecretsClient

# ── W&B login ──────────────────────────────────────────────────────────
os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
wandb.login()
print("W&B authenticated.")

# ── Device ─────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: 23f2002537 (23f2002537-indian-institute-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B authenticated.
Device: cuda


Configure paths, genres, and training hyperparameters.

In [2]:
# ── Constants ──────────────────────────────────────────────────────────
DATA_PATH  = "/kaggle/input/notebooks/noobshy/data-augmentation-image/train_data"
TEST_PATH  = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/mashups"
MODEL_DIR  = "/kaggle/working/models"
os.makedirs(MODEL_DIR, exist_ok=True)

genres      = ["blues","classical","country","disco","hiphop",
               "jazz","metal","pop","reggae","rock"]
genre_to_idx= {g: i for i, g in enumerate(genres)}
idx_to_genre= {i: g for g, i in genre_to_idx.items()}
NUM_CLASSES = 10
SR          = 22050
EPOCHS      = 25
BATCH_SIZE  = 32
VAL_SPLIT   = 0.15          # 15% held out for validation
WANDB_PROJECT = "23f2002537-t12026"

print(f"Genres: {genres}")

Genres: ['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock']


## 2. Dataset

Loads precomputed `.pt` mel tensors. Applies runtime SpecAugment during training only — validation set uses clean tensors for an unbiased eval signal.


In [3]:
class PrecomputedMelDataset(Dataset):
    """
    Loads precomputed mel-spectrogram tensors saved by the
    data-augmentation notebook.
    Applies optional runtime SpecAugment for extra diversity.
    """

    def __init__(self, data_path, augment=True, target_size=(128, 431)):
        self.data_path   = data_path
        self.files       = sorted(os.listdir(data_path))
        self.augment     = augment
        self.target_size = target_size   # (freq_bins, time_frames)

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        d   = torch.load(os.path.join(self.data_path, self.files[idx]))
        mel = d["mel"].float()           # (128, T)
        lbl = int(d["label"])

        # ── Pad or crop to a fixed width ────────────────────────────
        _, T = mel.shape
        target_T = self.target_size[1]
        if T < target_T:
            mel = F.pad(mel, (0, target_T - T))
        else:
            mel = mel[:, :target_T]

        # ── Runtime SpecAugment (training only) ─────────────────────
        if self.augment:
            mel = self._spec_augment(mel)

        return mel.unsqueeze(0), lbl    # → (1, 128, T)

    @staticmethod
    def _spec_augment(mel, freq_mask=20, time_mask=50, n=2):
        mel = mel.clone()
        F_bins, T = mel.shape
        for _ in range(n):
            f  = random.randint(1, freq_mask)
            f0 = random.randint(0, F_bins - f)
            mel[f0: f0 + f, :] = 0
            t  = random.randint(1, time_mask)
            t0 = random.randint(0, T - t)
            mel[:, t0: t0 + t] = 0
        return mel


# ── Build train / val splits ───────────────────────────────────────────
full_ds  = PrecomputedMelDataset(DATA_PATH, augment=True)
val_size = int(len(full_ds) * VAL_SPLIT)
trn_size = len(full_ds) - val_size
trn_ds, val_ds = random_split(full_ds, [trn_size, val_size],
                               generator=torch.Generator().manual_seed(42))
# Turn off augmentation for validation
val_ds.dataset.augment = False

trn_loader = DataLoader(trn_ds, batch_size=BATCH_SIZE, shuffle=True,
                        num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=2, pin_memory=True)

print(f"Train: {len(trn_ds):,}  |  Val: {len(val_ds):,}")
mel, lbl = next(iter(trn_loader))
print(f"Batch mel shape: {mel.shape}  label sample: {lbl[:4].tolist()}")

Train: 10,200  |  Val: 1,800
Batch mel shape: torch.Size([32, 1, 128, 431])  label sample: [9, 5, 6, 5]


## 3. Model Architectures

### Model 1 — GenreCNN
5-block custom CNN. Each block: `Conv2d → BN → ReLU → MaxPool`. Global average pooling replaces flatten to handle variable time dimensions.


In [4]:
class ConvBlock(nn.Module):
    """Conv2d → BN → ReLU → MaxPool"""
    def __init__(self, in_ch, out_ch, pool=True):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False)
        self.bn   = nn.BatchNorm2d(out_ch)
        self.pool = nn.MaxPool2d(2, 2) if pool else nn.Identity()

    def forward(self, x):
        return self.pool(F.relu(self.bn(self.conv(x))))


class GenreCNN(nn.Module):
    """
    5-block CNN built entirely from scratch.
    Input : (B, 1, 128, 431)
    Output: (B, NUM_CLASSES)
    """
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.encoder = nn.Sequential(
            ConvBlock(1,   32),   # → (B, 32,  64, 215)
            ConvBlock(32,  64),   # → (B, 64,  32, 107)
            ConvBlock(64,  128),  # → (B, 128, 16,  53)
            ConvBlock(128, 256),  # → (B, 256,  8,  26)
            ConvBlock(256, 512, pool=False),  # → (B, 512, 8, 26)
        )
        self.gap  = nn.AdaptiveAvgPool2d(1)  # → (B, 512, 1, 1)
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.head(self.gap(self.encoder(x)))


print("GenreCNN params:",
      sum(p.numel() for p in GenreCNN().parameters()) / 1e6, "M")

GenreCNN params: 1.70289 M


### Model 2 — EfficientNet-B0
Pretrained on ImageNet. First conv adapted from 3-channel to 1-channel by averaging pretrained weights. Input is upsampled to 224×224 at forward time.


In [5]:
class EfficientNetGenre(nn.Module):
    """
    EfficientNet-B0 backbone pretrained on ImageNet,
    with the final classifier replaced for 10-class genre prediction.
    Input : (B, 1, 128, 431)  — grayscale mel
    Output: (B, NUM_CLASSES)
    """
    def __init__(self, num_classes=NUM_CLASSES, freeze_backbone=False):
        super().__init__()
        backbone = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)

        # EfficientNet expects 3-channel input; adapt the first conv
        # by averaging the pretrained weights across channels
        orig_w    = backbone.features[0][0].weight.data   # (32, 3, 3, 3)
        new_conv  = nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1, bias=False)
        new_conv.weight.data = orig_w.mean(dim=1, keepdim=True)  # (32, 1, 3, 3)
        backbone.features[0][0] = new_conv

        if freeze_backbone:
            for p in backbone.parameters():
                p.requires_grad = False

        in_features = backbone.classifier[1].in_features
        backbone.classifier = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(in_features, num_classes)
        )
        self.net = backbone

    def forward(self, x):
        # Resize to 224×224 — EfficientNet's expected resolution
        x = F.interpolate(x, size=(224, 224), mode="bilinear", align_corners=False)
        return self.net(x)


print("EfficientNet-B0 params:",
      sum(p.numel() for p in EfficientNetGenre().parameters()) / 1e6, "M")

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 123MB/s] 


EfficientNet-B0 params: 4.019782 M


### Model 3 — CRNN
CNN front-end extracts per-frame frequency features; Bidirectional GRU captures temporal patterns across the clip. Final representation is the mean of all GRU output states.


In [6]:
class CRNN(nn.Module):
    """
    Convolutional Recurrent Neural Network.
    CNN: learns local frequency patterns per time-step.
    GRU: captures temporal evolution (musical rhythm) across the clip.
    Input : (B, 1, 128, T)
    Output: (B, NUM_CLASSES)
    """
    def __init__(self, num_classes=NUM_CLASSES, hidden_size=256, num_layers=2):
        super().__init__()
        # ── CNN front-end ──────────────────────────────────────────
        self.cnn = nn.Sequential(
            ConvBlock(1,  32, pool=True),   # (B, 32,  64, T/2)
            ConvBlock(32, 64, pool=True),   # (B, 64,  32, T/4)
            ConvBlock(64, 128, pool=True),  # (B, 128, 16, T/8)
        )
        # After CNN: frequency bins = 16, channels = 128
        # Merge freq × channel into a single feature vector per time-step
        cnn_out_dim = 128 * 16             # 2048

        # ── Bidirectional GRU ──────────────────────────────────────
        self.gru = nn.GRU(
            input_size=cnn_out_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.3,
            bidirectional=True
        )
        # ── Classifier head ────────────────────────────────────────
        self.head = nn.Sequential(
            nn.Linear(hidden_size * 2, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        # x : (B, 1, 128, T)
        x = self.cnn(x)                        # (B, 128, 16, T')
        B, C, F, T = x.shape
        x = x.permute(0, 3, 1, 2)             # (B, T', C, F)
        x = x.reshape(B, T, C * F)            # (B, T', 2048)
        out, _ = self.gru(x)                   # (B, T', hidden*2)
        out = out.mean(dim=1)                  # temporal average pooling
        return self.head(out)


print("CRNN params:",
      sum(p.numel() for p in CRNN().parameters()) / 1e6, "M")

CRNN params: 4.95153 M


## 4. Inference — Ensemble Submission

**Strategy:** Slide a 10s window (50% overlap, up to 8 windows) over each 30s test clip. Run EfficientNet + GenreCNN on each window, combine logits by weight, average across windows, take argmax.


In [8]:
def audio_to_mel_inference(path, sr=SR, segment_seconds=10):
    """
    Load audio, create mel spectrogram — identical pipeline to training.
    For long test clips, average over multiple 10-second windows.
    """
    segment_length = sr * segment_seconds
    y, _ = librosa.load(path, sr=sr, mono=True)

    # ── Multi-window averaging for long clips ────────────────────────
    hop = segment_length // 2      # 50% overlap
    starts = list(range(0, max(1, len(y) - segment_length + 1), hop))
    mels   = []

    for start in starts[:8]:       # cap at 8 windows to keep it fast
        chunk = y[start: start + segment_length]
        if len(chunk) < segment_length:
            chunk = librosa.util.fix_length(chunk, size=segment_length)
        mel = librosa.feature.melspectrogram(
            y=chunk, sr=sr, n_fft=2048, hop_length=512,
            n_mels=128, fmin=20, fmax=sr // 2
        )
        mel_db = librosa.power_to_db(mel, ref=np.max)
        mel_db = (mel_db - mel_db.mean()) / (mel_db.std() + 1e-6)
        mels.append(mel_db.astype(np.float32))

    # Stack all windows and average predictions over them
    return mels   # list of (128, T) arrays

Load models from Hub and run inference on the test set.

In [9]:
TARGET_T = 431   # fixed time dimension used during training

def predict_file(path, eff_model, cnn_model,
                 eff_weight=0.4, cnn_weight=0.6):
    mels = audio_to_mel_inference(path)
    all_logits = []

    for mel_arr in mels:
        T = mel_arr.shape[1]
        if T < TARGET_T:
            mel_arr = np.pad(mel_arr, ((0, 0), (0, TARGET_T - T)))
        else:
            mel_arr = mel_arr[:, :TARGET_T]

        tensor = torch.tensor(mel_arr).float().unsqueeze(0).unsqueeze(0).to(device)

        with torch.no_grad():
            eff_logits = eff_model(tensor)
            cnn_logits = cnn_model(tensor)

        combined = eff_weight * eff_logits + cnn_weight * cnn_logits
        all_logits.append(combined)

    avg_logits = torch.stack(all_logits).mean(dim=0)
    return torch.argmax(avg_logits, dim=1).item()

# ── Run inference ───────────────────────────────────────────────────────
KAGGLE_USERNAME = UserSecretsClient().get_secret("Kaggle_username")

# ── Load CNN ───────────────────────────────────────────────────────────
cnn_model = GenreCNN(NUM_CLASSES).to(device)
cnn_model.load_state_dict(torch.load(
    "/kaggle/input/models/noobshy/genrecnn-messy-mashup-shy/pytorch/default/1/GenreCNN_best.pth",
    map_location=device
))
cnn_model.eval()

# ── Load EfficientNet ──────────────────────────────────────────────────
eff_model = EfficientNetGenre(NUM_CLASSES).to(device)
eff_model.load_state_dict(torch.load(
    "/kaggle/input/models/noobshy/efficientnet-messy-mashup-shy/pytorch/default/1/EfficientNet_best.pth",
    map_location=device
))
eff_model.eval()

test_files = sorted(os.listdir(TEST_PATH))
ids, predictions = [], []

for fname in tqdm(test_files, desc="Inference"):
    fpath = os.path.join(TEST_PATH, fname)
    pred  = predict_file(fpath, eff_model, cnn_model)
    predictions.append(idx_to_genre[pred])
    ids.append(fname.replace(".wav", "").replace("song", ""))

print(f"Predicted {len(predictions)} files.")

Inference: 100%|██████████| 3020/3020 [08:45<00:00,  5.75it/s]

Predicted 3020 files.


## 5. Save Submission

In [10]:
# ── Save submission ─────────────────────────────────────────────────────
submission = pd.DataFrame({"id": ids, "genre": predictions})
submission.to_csv("submission.csv", index=False)

print("submission.csv saved!")
print(submission["genre"].value_counts())
submission.head(10)

submission.csv saved!
genre
rock         326
pop          324
classical    321
reggae       321
metal        317
hiphop       315
disco        297
jazz         295
blues        254
country      250
Name: count, dtype: int64


,id,genre
0,0001,pop
1,0002,classical
2,0003,disco
3,0004,metal
4,0005,country
5,0006,pop
6,0007,rock
7,0008,pop
8,0009,pop
9,0010,disco
